In [1]:
import numpy as np
from itertools import chain
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import pdist

import json
from scipy.stats import zscore
from utils import extract_hrv_features

ages_table_path = 'edades.xlsx'
ages_table = pd.read_excel(ages_table_path)

I0000 00:00:1783976439.578864  734886 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783976441.675349  734886 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# create a dictionary of groups by ages
dict_ages = {}

for age in ages_table['ages'].unique():
    codes_list = ages_table.loc[ages_table['ages'] == age, "codes"].tolist()
    for i in range(len(codes_list)):
        raw_code = str(codes_list[i])
        if len(raw_code) == 1:
            raw_code = '00' + raw_code
        elif len(raw_code) == 2:
            raw_code = '0' + raw_code
        raw_code += '.txt'
        
        codes_list[i] = raw_code
    dict_ages[age] = codes_list

**Normalization parameters by age**

Note: let $x$ be age (years). For RR mean, RR standard deviation, and pNN50 use the formulas below.

RR mean:

$$
\text{RR\_mean} = 505 \cdot x^{0.122}
$$

For RR standard deviation and pNN50 use the age-dependent formulas:

If $x \le 12$:

$$
\text{RR\_std} = 80 \cdot x^{0.26} \\,
\text{pNN50} = 0.037 \cdot x^{0.78}
$$

If $x > 12$:

$$
\text{RR\_std} = 290 \cdot x^{-0.2} \\,
\text{pNN50} = 5 \cdot x^{-1.1}
$$

Python implementation:

```python
# x = age (years)
RR_mean = 505 * x**0.122
if x <= 12:
    RR_std = 80 * x**0.26
    pNN50 = 0.037 * x**0.78
else:
    RR_std = 290 * x**(-0.2)
    pNN50 = 5 * x**(-1.1)
```

In [3]:
# define testing subjects
range_groups = np.array([[0, 1], [1, 12], [25, 40], [40, 65], [65, 74]])
np.random.seed(7)
test_percent = 0.1
# select 10% subjects of each group to test
subjects_test = []

for interval in range_groups:
    
    start, end = interval
    subjects = ages_table.loc[(ages_table['ages'] >= start) & (ages_table['ages'] < end), "codes"].tolist()
    quantity = int(test_percent * len(subjects))
    files_test = list(np.random.choice(subjects, size=quantity, replace=False))
    
    for i in range(len(files_test)):
        raw_code = str(files_test[i])
        if len(raw_code) == 1:
            raw_code = '00' + raw_code
        elif len(raw_code) == 2:
            raw_code = '0' + raw_code
        raw_code += '.txt'
        files_test[i] = raw_code

    subjects_test.append(files_test)

In [16]:
print(subjects_test)

[['4060.txt', '4041.txt', '4026.txt', '4090.txt', '4065.txt', '4018.txt', '4019.txt'], ['4111.txt', '4025.txt', '407.txt', '416.txt'], ['16265.txt'], ['16483.txt'], ['nsr009RRcl.txt']]


In [4]:
flat_subjs = list(chain.from_iterable(subjects_test))

In [6]:
print(flat_subjs)

['4060.txt', '4041.txt', '4026.txt', '4090.txt', '4065.txt', '4018.txt', '4019.txt', '4111.txt', '4025.txt', '407.txt', '416.txt', '16265.txt', '16483.txt', 'nsr009RRcl.txt']
